In [0]:
# Import core runnable classes for building data pipelines and workflows
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnablePassthrough

# Import PromptTemplate for prompt engineering with language models
from langchain.prompts import PromptTemplate

# Import Databricks-specific chat LLM integration
from databricks_langchain import ChatDatabricks

# Import hashlib for cryptographic hashing functions
import hashlib

# Import re for regular expression operations
import re

In [0]:
def encrypt_password(password: str) -> str:
    """Encrypt the given password using SHA-256."""
    return hashlib.sha256(password.encode("UTF-8")).hexdigest()

Pass your parameters through `invoke()` to call a runnable type object.
- The `invoke()` method executes the underlying function or sequence with the provided input.
- `RunnableLambda` is like a wrapper on top of a function that is called through `invoke()`
- Use `invoke()` to trigger the computation and get the result.

In [0]:
encrypt_password = RunnableLambda(encrypt_password)
print(encrypt_password.invoke("hello123"))
print(encrypt_password.batch(["hello123", "world123"]))

### RunnableLambda
- A RunnableLambda class is a wrapper applied on top of a function. It is used to apply transformations to inputs

In [0]:
def download_csv_files():
    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]

    def load_all_csv_files(file_list):
        for file in file_list:
            print("Data load completed for file:", file)

    return RunnableLambda(list_all_csv_files) | RunnableLambda(load_all_csv_files)


# A RunnableLambda class is a wrapper applied on top of a function. It is used to apply transformations to inputs
# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.

files = download_csv_files()
print(files.invoke("http://example.com/data"))

### RunnableSequence
- RunnableSequence is used to run runnable instances in a sequential order so that output from one runnable is carried to the next runnable instance

In [0]:
def load_all_csv_files():
    """
    In the first
    RunnableLambda(lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]})
    we're:
        1. Calling list_all_csv_files(x["url"], "n": x["n"]) to get the file list using only the url from the input dict. "n": x["n"] in the input dict is never used in this function. But it is carried forward for next function call. We returned a dictionary that is further used down the lane.
        2. That "n": x["n"] is used to call chunkify like chunkify(d["file_list"], d["n"])
           This output dict (with file_list and n) is then passed to the second RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])), which uses both values to create the chunks.
    """

    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]

    def chunkify(file_list, n):
        result = []
        for file in file_list:
            for i in range(n):
                result.append(file.replace(".csv", f"_part{i}.csv"))
        return result

    def load_all_csv_files(file_list):
        for file in file_list:
            print(f"Data load completed for file:{file}")

    return RunnableSequence(
        RunnableLambda(
            lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]}
        ),
        RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])),
        RunnableLambda(load_all_csv_files),
    )


# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.

files = load_all_csv_files()
print(files.invoke({"url": "http://example.com/data", "n": 2}))

### Build a pipeline using RunnableSequence 
### & RunnableLambda

In [0]:
def generate_odd_number(n: int) -> list[int]:
    """Generate a list of odd numbers up to n."""
    return [i for i in range(n) if i % 2 != 0]


def sum_of_odd_numbers(odd_numbers: list[int]) -> int:
    """Calculate the sum of a list of odd numbers."""
    return sum(odd_numbers)


def check_palindrome(s: int) -> bool:
    s = str(s)
    return s == s[::-1]


output = RunnableSequence(
    first=RunnableLambda(generate_odd_number),
    middle=[RunnableLambda(sum_of_odd_numbers)],
    last=RunnableLambda(check_palindrome),
)

# output = RunnableSequence(
#     first = RunnableLambda(generate_odd_number),
#     last = RunnableLambda(sum_of_odd_numbers)
# )

print(output.invoke(22))  # 121  is a palindrome
print(output.invoke(10))  # 25 is not a palindrome

In [0]:
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2500)

prompt = PromptTemplate.from_template(
    template="Give me the steps create rest API connection in databricks"
)

formatted_prompt = prompt.format()

# result = llm.invoke(formatted_prompt)

# To pass a prompt in a runnable instance, use RunnableLambda and provide the prompt as input.
# If the prompt does not require any parameters, pass None to the invoke() method.

chain = RunnableLambda(lambda x: formatted_prompt) | llm
print(chain.invoke(None).content)

In [0]:
# Create a prompt template with parameters {n} and {year}
prompt = PromptTemplate.from_template(template = "Give me top {n} events on year {year}")

# Pass a dictionary of parameters to the prompt using RunnableLambda.
# The lambda unpacks the input dict and formats the prompt with the provided values.
chain = RunnableLambda(lambda x: prompt.format(**x)) | llm

print(chain.invoke({"n": 5, "year": 2025}).content)

### RunnablePassthrough

In [0]:
from datetime import datetime


def get_today(day):
    return datetime.today().strftime("%y")


prompt = PromptTemplate.from_template(template="Give me top {n} events on {day}")

# RunnablePassthrough() is used to pass the input value directly to the next step without any transformation.
# Here, it passes the input value as 'n' to the prompt template.
chain = {"day": RunnableLambda(get_today), "n": RunnablePassthrough()} | prompt | llm
print(chain.invoke(2).content)

### Build a pipeline using RunnableLambda 
### & RunnablePassthrough

In [0]:
def get_year(date: str) -> int:
    """Extract the year from a date string."""
    return int(re.search(r"\d{4}", date).group())

prompt = PromptTemplate.from_template(template = "Give me top {n} events on year {year}")

# RunnablePassthrough.assign can be used to override or add parameters in the input dictionary.
# It allows you to specify new key-value pairs, where the value can be a function or a Runnable.
# The assigned values will override any existing keys in the input dict or add new ones.
# In this example:
#   - "n" is overridden by extracting it from the input dict (x["n"])
#   - "year" is set by extracting the year from the "date" field using get_year
# The resulting dict is then passed to the prompt template and LLM.

chain = RunnablePassthrough.assign(n = lambda x: x["n"], year = RunnableLambda(lambda x: get_year(x["date"]))) | prompt | llm

print(chain.invoke({"n": 5, "date": "2025-01-01"}).content)